# 04 - Tuning de Random Forest

## Predicción de Alertas de Demanda Operativa - DNA (UMSS 2026)

### Afinamiento de hiperparámetros y evaluación temporal

- **Train:** 2023
- **Validation:** 2024
- **Test Hold-Out:** 2025

En este notebook se realiza:

1. Optimización de hiperparámetros de Random Forest.
2. Validación cruzada temporal mediante `TimeSeriesSplit`.
3. Selección del umbral de clasificación utilizando Validation 2024.
4. Evaluación final sobre Test 2025.
5. Generación de la matriz de confusión.
6. Exportación del modelo optimizado.


# Importaciones

In [ ]:
from pathlib import Path
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from sklearn.model_selection import (
    TimeSeriesSplit,
    GridSearchCV
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline


# Configuración de directorios

In [ ]:
project_root = Path.cwd()

data_dir = project_root / "data" / "processed"
results_dir = project_root / "results"
models_dir = project_root / "models_saved"

results_dir.mkdir(
    parents=True,
    exist_ok=True
)

models_dir.mkdir(
    parents=True,
    exist_ok=True
)

print(f"Directorio del proyecto: {project_root}")
print(f"Directorio de datos: {data_dir}")
print(f"Directorio de resultados: {results_dir}")
print(f"Directorio de modelos: {models_dir}")


# Cargar los datos

In [ ]:
print(
    "Cargando bloques: "
    "Train (2023), Validation (2024) y Test Final (2025)..."
)

X_train = pd.read_csv(
    data_dir / "X_train.csv"
)

y_train = pd.read_csv(
    data_dir / "y_train.csv"
).values.ravel()

X_val = pd.read_csv(
    data_dir / "X_val.csv"
)

y_val = pd.read_csv(
    data_dir / "y_val.csv"
).values.ravel()

X_test = pd.read_csv(
    data_dir / "X_test.csv"
)

y_test = pd.read_csv(
    data_dir / "y_test.csv"
).values.ravel()

print("\nDatos cargados correctamente.")

print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")

print(f"X_val:   {X_val.shape}")
print(f"y_val:   {y_val.shape}")

print(f"X_test:  {X_test.shape}")
print(f"y_test:  {y_test.shape}")


# Distribución de clases

In [ ]:
print("Distribución de clases - Train")

train_distribution = (
    pd.Series(y_train)
    .value_counts()
    .sort_index()
)

display(train_distribution)


In [ ]:
print("Distribución porcentual - Train")

display(
    pd.Series(y_train)
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)


# Crear Pipeline

In [ ]:
pipeline = Pipeline(
    [
        (
            "clf",
            RandomForestClassifier(
                random_state=42,
                class_weight="balanced"
            )
        )
    ]
)

pipeline


# Grid de hiperparámetros

In [ ]:
param_grid = {
    "clf__n_estimators": [
        100,
        200,
        300
    ],
    "clf__max_depth": [
        5,
        8,
        12,
        None
    ],
    "clf__min_samples_leaf": [
        1,
        3,
        5
    ],
    "clf__min_samples_split": [
        2,
        5
    ]
}

param_grid


In [ ]:
from itertools import product

total_combinations = np.prod([
    len(values)
    for values in param_grid.values()
])

print(
    f"Combinaciones de hiperparámetros: "
    f"{total_combinations}"
)

print(
    f"Particiones temporales: 3"
)

print(
    f"Entrenamientos aproximados del GridSearch: "
    f"{total_combinations * 3}"
)


# Validación cruzada temporal

In [ ]:
tscv = TimeSeriesSplit(
    n_splits=3
)

print(tscv)


# Entrenamiento y GridSearch

In [ ]:
print(
    "\nIniciando GridSearch "
    "para Random Forest..."
)

start_time = time.time()

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=tscv,
    scoring="f1",
    n_jobs=-1
)

grid.fit(
    X_train,
    y_train
)

train_time = round(
    time.time() - start_time,
    2
)

print(
    f"\n[OK] Entrenamiento finalizado "
    f"en {train_time} segundos"
)

print(
    "\nMejores hiperparámetros:"
)

print(
    grid.best_params_
)

print(
    f"\nMejor F1-Score promedio "
    f"en validación cruzada: "
    f"{grid.best_score_:.4f}"
)


# Obtener el mejor modelo

In [ ]:
best_pipeline = grid.best_estimator_

best_pipeline


# Analizar resultados del GridSearch

In [ ]:
df_grid = pd.DataFrame(
    grid.cv_results_
)

df_grid_results = df_grid[
    [
        "param_clf__n_estimators",
        "param_clf__max_depth",
        "param_clf__min_samples_leaf",
        "param_clf__min_samples_split",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].sort_values(
    "rank_test_score"
)

display(
    df_grid_results
)


# Probabilidades en Validation 2024

In [ ]:
y_val_proba = best_pipeline.predict_proba(
    X_val
)[:, 1]

print(
    "Probabilidades generadas "
    "para Validation 2024."
)

print(
    f"Probabilidad mínima: "
    f"{y_val_proba.min():.4f}"
)

print(
    f"Probabilidad máxima: "
    f"{y_val_proba.max():.4f}"
)


Selección del umbral

In [ ]:
thresholds = [
    0.30,
    0.35,
    0.40,
    0.45,
    0.50
]

threshold_results = []

for th in thresholds:

    y_val_pred = (
        y_val_proba >= th
    ).astype(int)

    f1_v = f1_score(
        y_val,
        y_val_pred,
        zero_division=0
    )

    rec_v = recall_score(
        y_val,
        y_val_pred,
        zero_division=0
    )

    prec_v = precision_score(
        y_val,
        y_val_pred,
        zero_division=0
    )

    threshold_results.append(
        {
            "Umbral (th)": th,
            "Recall": round(rec_v, 4),
            "Precision": round(prec_v, 4),
            "F1-Score": round(f1_v, 4)
        }
    )

df_threshold_val = pd.DataFrame(
    threshold_results
)

display(
    df_threshold_val
)


# Seleccionar el umbral

In [ ]:
best_row = df_threshold_val.loc[
    df_threshold_val["F1-Score"].idxmax()
]

best_th = float(
    best_row["Umbral (th)"]
)

best_f1_val = float(
    best_row["F1-Score"]
)

print(
    f"Umbral seleccionado en "
    f"Validation 2024: {best_th:.2f}"
)

print(
    f"F1-Score en Validation: "
    f"{best_f1_val:.4f}"
)


# Gráfica F1 vs umbral

In [ ]:
plt.figure(
    figsize=(8, 5)
)

plt.plot(
    df_threshold_val["Umbral (th)"],
    df_threshold_val["F1-Score"],
    marker="o",
    linewidth=2,
    label="F1-Score"
)

plt.axvline(
    best_th,
    color="red",
    linestyle="--",
    label=(
        f"Umbral seleccionado = "
        f"{best_th:.2f}"
    )
)

plt.xlabel(
    "Umbral de clasificación"
)

plt.ylabel(
    "F1-Score"
)

plt.title(
    "Selección de Umbral - "
    "Validation 2024"
)

plt.grid(True)

plt.legend()

plt.tight_layout()

plt.show()


# Probabilidades del Test 2025

In [ ]:
y_test_proba = best_pipeline.predict_proba(
    X_test
)[:, 1]

print(
    "Probabilidades generadas "
    "para Test 2025."
)


# Evaluación en Test 2025

In [ ]:
results_test = []

for th in thresholds:

    y_test_pred = (
        y_test_proba >= th
    ).astype(int)

    acc = accuracy_score(
        y_test,
        y_test_pred
    )

    prec = precision_score(
        y_test,
        y_test_pred,
        zero_division=0
    )

    rec = recall_score(
        y_test,
        y_test_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_test_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        y_test_proba
    )

    results_test.append(
        {
            "Modelo": (
                f"Random Forest "
                f"(th = {th:.2f})"
            ),
            "Umbral (th)": th,
            "Accuracy": round(acc, 4),
            "Precision": round(prec, 4),
            "Recall": round(rec, 4),
            "F1-Score": round(f1, 4),
            "ROC-AUC": round(roc_auc, 4),
            "Seleccionado_en_Val": (
                th == best_th
            )
        }
    )

df_results = pd.DataFrame(
    results_test
)

display(
    df_results
)


# Resultado correspondiente al umbral seleccionado

In [ ]:
df_resultado_seleccionado = df_results[
    df_results["Seleccionado_en_Val"]
]

display(
    df_resultado_seleccionado
)


# Guardar resultados

In [ ]:
output_path = (
    results_dir /
    "resultados_rf_tuning_umbrales.csv"
)

df_results.to_csv(
    output_path,
    index=False
)

print(
    "[OK] Resultados guardados en:"
)

print(
    output_path
)


# Matriz de confusión

In [ ]:
y_test_pred_best = (
    y_test_proba >= best_th
).astype(int)

cm = confusion_matrix(
    y_test,
    y_test_pred_best
)

fig, ax = plt.subplots(
    figsize=(6, 5)
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Inactivo (0)",
        "Demanda Activa (1)"
    ]
)

disp.plot(
    cmap="Blues",
    ax=ax,
    values_format="d"
)

plt.title(
    f"Matriz de Confusión - "
    f"Test 2025 (th = {best_th:.2f})",
    fontsize=11,
    fontweight="bold"
)

plt.xlabel(
    "Predicción del Modelo",
    fontsize=10
)

plt.ylabel(
    "Clase Real (Observada)",
    fontsize=10
)

plt.grid(False)

plt.tight_layout()

plt.show()


# Guardar matriz de confusión

In [ ]:
cm_path = (
    results_dir /
    "matriz_confusion_rf_opt.png"
)

fig, ax = plt.subplots(
    figsize=(6, 5)
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Inactivo (0)",
        "Demanda Activa (1)"
    ]
)

disp.plot(
    cmap="Blues",
    ax=ax,
    values_format="d"
)

plt.title(
    f"Matriz de Confusión - "
    f"Test 2025 (th = {best_th:.2f})",
    fontsize=11,
    fontweight="bold"
)

plt.xlabel(
    "Predicción del Modelo",
    fontsize=10
)

plt.ylabel(
    "Clase Real (Observada)",
    fontsize=10
)

plt.grid(False)

plt.tight_layout()

plt.savefig(
    cm_path,
    dpi=300
)

plt.show()

plt.close()

print(
    f"[OK] Gráfico guardado en:\n"
    f"{cm_path}"
)


# Guardar el modelo

In [ ]:
pkl_path = (
    models_dir /
    "random_forest_tuned_model.pkl"
)

joblib.dump(
    best_pipeline,
    pkl_path
)

print(
    "[OK] Modelo exportado exitosamente:"
)

print(
    pkl_path
)


# Resumen final

In [ ]:
print("=" * 85)
print("RESUMEN FINAL - RANDOM FOREST")
print("=" * 85)

print(
    "\nMejores hiperparámetros:"
)

print(
    grid.best_params_
)

print(
    f"\nF1-Score promedio CV: "
    f"{grid.best_score_:.4f}"
)

print(
    f"\nTiempo de entrenamiento: "
    f"{train_time} segundos"
)

print(
    f"\nUmbral seleccionado en "
    f"Validation 2024: "
    f"{best_th:.2f}"
)

print(
    f"F1-Score Validation: "
    f"{best_f1_val:.4f}"
)

print(
    "\nResultado correspondiente "
    "en Test 2025:"
)

display(
    df_resultado_seleccionado
)
